# Python Data Structures for Data Engineering

## Purpose of This Notebook

This notebook teaches you **essential Python data structure operations** that you'll use daily as a Data Engineer:

| Skill | Real-World Use Case |
|-------|--------------------|
| Flatten/Unflatten Dicts | Parse nested JSON from APIs, prepare data for Spark |
| Log Parsing | Analyze application logs, count errors |
| Grouping Data | Find duplicates, aggregate data |
| Pure Python Joins | Understand join logic before using Spark/Pandas |
| Generators | Process files larger than memory |
| Sliding Windows | Calculate rolling metrics (moving averages) |

---

In [ ]:
# IMPORTS - These are Python's built-in data structure helpers
from collections import defaultdict, Counter, deque  # Specialized containers
from typing import List, Dict, Any, Optional, Iterator  # Type hints for clarity
import csv  # Read/write CSV files
from io import StringIO  # Treat strings as files (for demos)
import copy  # For deep copying nested structures

---
## 1. Flatten a Nested Dictionary

### Why is this useful?
- **API responses** often return deeply nested JSON
- **Spark/databases** work better with flat structures
- **Log analysis** often requires flattening nested fields

### Example:
```python
# Input (nested):
{"user": {"name": "Alice", "address": {"city": "NYC"}}}

# Output (flat):
{"user.name": "Alice", "user.address.city": "NYC"}
```

In [ ]:
def flatten_dict(d: dict, parent_key: str = '', sep: str = '.') -> dict:
    """
    Recursively flatten a nested dictionary.
    
    HOW IT WORKS:
    1. Loop through each key-value pair
    2. If value is a dict -> recurse deeper
    3. If value is not a dict -> add to result with concatenated key
    
    Args:
        d: The dictionary to flatten
        parent_key: Used internally to track the key path (e.g., "user.address")
        sep: Separator between keys (default is ".")
    
    Returns:
        Flattened dictionary with dot-separated keys
    """
    items = []  # Will store (key, value) tuples
    
    for k, v in d.items():
        # Build the new key: "parent.child" or just "key" if no parent
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        
        if isinstance(v, dict):
            # VALUE IS A DICT: Recurse into it
            # Example: {"a": {"b": 1}} -> flatten_dict({"b": 1}, "a")
            nested_items = flatten_dict(v, new_key, sep)
            items.extend(nested_items.items())  # Add all nested results
        else:
            # VALUE IS NOT A DICT: Add to results
            # Example: ("a.b", 1)
            items.append((new_key, v))
    
    return dict(items)  # Convert list of tuples to dict

In [ ]:
# DEMO: Flatten a nested dictionary
nested_data = {
    "user": {
        "name": "Alice",
        "address": {
            "city": "NYC",
            "zip": "10001"
        }
    },
    "active": True
}

print("BEFORE (nested):")
print(nested_data)

flat_data = flatten_dict(nested_data)

print("\nAFTER (flat):")
for key, value in flat_data.items():
    print(f"  {key}: {value}")

# STEP-BY-STEP EXECUTION:
# 1. Start with {"user": {...}, "active": True}
# 2. "user" is a dict -> recurse with parent_key="user"
#    - "name" is "Alice" (not dict) -> add ("user.name", "Alice")
#    - "address" is a dict -> recurse with parent_key="user.address"
#      - "city" is "NYC" -> add ("user.address.city", "NYC")
#      - "zip" is "10001" -> add ("user.address.zip", "10001")
# 3. "active" is True (not dict) -> add ("active", True)

---
## 2. Unflatten a Dictionary (Reverse Operation)

### Why is this useful?
- Convert **flat database rows** back to nested JSON for APIs
- Reconstruct **original structure** after processing

In [2]:
def unflatten_dict(d: dict, sep: str = '.') -> dict:
    """
    Convert a flat dictionary back to nested structure.
    
    HOW IT WORKS:
    1. Split each key by separator ("user.address.city" -> ["user", "address", "city"])
    2. Navigate/create nested dicts for each part
    3. Set the value at the deepest level
    """
    result = {}  # Will be built up as nested dict
    
    for key, value in d.items():
        parts = key.split(sep)  # ["user", "address", "city"]
        
        # Navigate to the right nested level
        current = result
        for part in parts[:-1]:  # All parts except the last
            # setdefault: get value if exists, else create empty dict
            current = current.setdefault(part, {})
        
        # Set the final value
        current[parts[-1]] = value  # parts[-1] is the last key
    
    return result

In [3]:
# DEMO: Unflatten back to nested
flat_data = {
    "user.name": "Alice",
    "user.address.city": "NYC",
    "user.address.zip": "10001",
    "active": True
}

print("BEFORE (flat):")
print(flat_data)

nested_data = unflatten_dict(flat_data)

print("\nAFTER (nested):")
import json
print(json.dumps(nested_data, indent=2))

BEFORE (flat):
{'user.name': 'Alice', 'user.address.city': 'NYC', 'user.address.zip': '10001', 'active': True}

AFTER (nested):
{
  "user": {
    "name": "Alice",
    "address": {
      "city": "NYC",
      "zip": "10001"
    }
  },
  "active": true
}


---
## 3. defaultdict - Auto-Creating Nested Structures

### Why is this useful?
- **No KeyError** when accessing missing keys
- **Auto-initialize** counters, lists, or nested dicts
- **Cleaner code** - no need to check if key exists

In [4]:
# PROBLEM: Regular dict raises KeyError for missing keys
regular_dict = {}
try:
    regular_dict["errors"] += 1  # KeyError! "errors" doesn't exist
except KeyError as e:
    print(f"❌ Regular dict error: KeyError for {e}")

# SOLUTION: defaultdict auto-creates missing keys
error_counts = defaultdict(int)  # Missing keys default to 0
error_counts["connection"] += 1  # Works! Creates key with 0, then adds 1
error_counts["connection"] += 1  # Now it's 2
error_counts["timeout"] += 1     # New key "timeout" = 1

print(f"\n✅ defaultdict(int): {dict(error_counts)}")

❌ Regular dict error: KeyError for 'errors'

✅ defaultdict(int): {'connection': 2, 'timeout': 1}


In [ ]:
# EXAMPLE: Group items by category using defaultdict(list)
products = [
    {"name": "iPhone", "category": "Electronics"},
    {"name": "Shirt", "category": "Clothing"},
    {"name": "MacBook", "category": "Electronics"},
    {"name": "Jeans", "category": "Clothing"},
]

# Group by category
by_category = defaultdict(list)  # Missing keys default to empty list []
for product in products:
    category = product["category"]
    by_category[category].append(product["name"])  # No need to check if key exists!

print("Products grouped by category:")
for category, items in by_category.items():
    print(f"  {category}: {items}")

---
## 4. Log Parsing - Count Errors Per User

### Real-World Scenario:
You have application logs and need to:
- Find which users have the most errors
- Aggregate error counts for monitoring dashboards

In [ ]:
def count_errors_per_user(logs: List[str]) -> Dict[str, int]:
    """
    Parse logs and count ERROR entries per user.
    
    Log format: "TIMESTAMP USER_ID LEVEL MESSAGE"
    
    HOW IT WORKS:
    1. Split each log line into parts
    2. Check if level is "ERROR"
    3. Increment counter for that user
    """
    # defaultdict(int) starts missing keys at 0
    error_counts = defaultdict(int)
    
    for log in logs:
        # Split into: [timestamp, user_id, level, message...]
        # maxsplit=3 means split into at most 4 parts
        parts = log.split(maxsplit=3)
        
        if len(parts) >= 3:
            timestamp, user_id, level = parts[0], parts[1], parts[2]
            
            if level == "ERROR":
                error_counts[user_id] += 1  # Increment user's error count
    
    return dict(error_counts)

In [ ]:
# DEMO: Parse application logs
logs = [
    "2023-01-01T10:00:00 user1 INFO Login successful",
    "2023-01-01T10:01:00 user2 ERROR Database connection failed",
    "2023-01-01T10:02:00 user1 ERROR File not found",
    "2023-01-01T10:03:00 user3 WARN Low memory",
    "2023-01-01T10:04:00 user2 ERROR Timeout",
    "2023-01-01T10:05:00 user2 ERROR Permission denied",
]

print("Sample logs:")
for log in logs[:3]:
    print(f"  {log}")
print("  ...")

error_counts = count_errors_per_user(logs)

print("\n📊 Error counts per user:")
for user, count in sorted(error_counts.items(), key=lambda x: -x[1]):
    print(f"  {user}: {count} errors")

---
## 5. Generators - Process Large Files Without Memory Issues

### The Problem:
You have a **10GB CSV file**. Loading it all into memory will crash your program!

### The Solution: Generators
- Process **one row at a time**
- Uses **constant memory** regardless of file size
- `yield` returns a value and **pauses** the function

In [ ]:
def process_large_csv(csv_content: str) -> Iterator[Dict]:
    """
    Generator that yields one row at a time.
    
    KEY CONCEPT: yield vs return
    - return: Function ends, all data returned at once
    - yield: Function pauses, returns one item, resumes when next() called
    
    MEMORY: O(1) - constant memory regardless of file size!
    """
    # csv.DictReader reads one row at a time (generator-like)
    reader = csv.DictReader(StringIO(csv_content))
    
    for row in reader:
        # Process/transform the row
        row['amount'] = float(row['amount'])  # Convert string to float
        row['processed'] = True  # Add new field
        
        yield row  # PAUSE here, return this row, resume on next iteration

In [ ]:
# DEMO: Process CSV row by row
csv_data = """id,name,amount
1,Alice,100.50
2,Bob,200.75
3,Charlie,300.00
4,Diana,150.25"""

print("Processing rows one at a time (generator):")
print("="*50)

# The generator doesn't load all rows - it processes ONE AT A TIME
for row in process_large_csv(csv_data):
    print(f"  Processing: {row}")
    # In real code, you'd write to database, transform, etc.

print("\n💡 Memory usage: Only 1 row in memory at any time!")

In [ ]:
# COMPARE: List vs Generator memory usage
import sys

# List: Stores ALL items in memory
list_data = [x for x in range(10000)]
list_memory = sys.getsizeof(list_data)

# Generator: Creates items on-demand
gen_data = (x for x in range(10000))
gen_memory = sys.getsizeof(gen_data)

print(f"Memory for 10,000 items:")
print(f"  List:      {list_memory:,} bytes")
print(f"  Generator: {gen_memory:,} bytes")
print(f"  Savings:   {(list_memory - gen_memory) / list_memory * 100:.1f}%")

---
## 6. Sliding Window - Moving Average

### Real-World Use Cases:
- **Stock prices**: 7-day moving average
- **Metrics**: Rolling average of response times
- **Sensors**: Smooth out noisy readings

In [ ]:
def moving_average(values: List[float], window_size: int) -> List[float]:
    """
    Calculate moving average using a sliding window.
    
    HOW IT WORKS:
    1. Use deque (double-ended queue) with max size = window_size
    2. As we add new values, old values automatically drop off
    3. Calculate average when window is full
    
    TIME: O(n) - single pass through data
    SPACE: O(k) where k = window_size
    """
    if not values or window_size <= 0:
        return []
    
    result = []
    # deque with maxlen automatically removes oldest item when full
    window = deque(maxlen=window_size)
    window_sum = 0
    
    for val in values:
        # Add new value
        window_sum += val
        window.append(val)
        
        # Only calculate average when window is full
        if len(window) == window_size:
            average = window_sum / window_size
            result.append(average)
            
            # Subtract the value that will be removed next iteration
            window_sum -= window[0]
    
    return result

In [ ]:
# DEMO: Calculate 3-day moving average
daily_sales = [100, 120, 110, 130, 125, 140, 135]
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

print("Daily Sales:")
for day, sales in zip(days, daily_sales):
    print(f"  {day}: ${sales}")

averages = moving_average(daily_sales, window_size=3)

print("\n3-Day Moving Average:")
# First average starts at day 3 (need 3 days of data)
for i, avg in enumerate(averages):
    print(f"  {days[i]}-{days[i+2]}: ${avg:.2f}")

# STEP-BY-STEP:
# Window [100, 120, 110] -> avg = 110.0 (Mon-Wed)
# Window [120, 110, 130] -> avg = 120.0 (Tue-Thu)
# ...

---
## 7. Implement Left Join in Pure Python

### Why Learn This?
- Understand **how JOINs actually work** before using Spark/Pandas
- Useful when you can't use external libraries
- Common interview question!

In [ ]:
def left_join(left: List[Dict], right: List[Dict], key: str) -> List[Dict]:
    """
    LEFT JOIN: Return all rows from left table, with matching rows from right.
    
    HOW IT WORKS:
    1. Build a lookup table (hash map) from the right table - O(m)
    2. For each left row, look up matching right row - O(n)
    3. Merge if found, otherwise keep left row as-is
    
    TIME: O(n + m)  - very efficient!
    SPACE: O(m) for the lookup table
    """
    # STEP 1: Build lookup table from right side
    # This makes finding matches O(1) instead of O(m)
    right_lookup = {}
    for item in right:
        right_lookup[item[key]] = item
    
    # STEP 2: Join each left row with its matching right row
    result = []
    for left_item in left:
        joined = left_item.copy()  # Don't modify original
        
        # Look up matching right row
        right_item = right_lookup.get(left_item[key])
        
        if right_item:
            # MATCH FOUND: Merge the two rows
            for k, v in right_item.items():
                if k != key:  # Don't duplicate the join key
                    joined[k] = v
        # If no match: left row returned as-is (LEFT JOIN behavior)
        
        result.append(joined)
    
    return result

In [ ]:
# DEMO: Join users with their departments
users = [
    {"id": 1, "name": "Alice"},
    {"id": 2, "name": "Bob"},
    {"id": 3, "name": "Charlie"},  # No matching department!
]

departments = [
    {"id": 1, "department": "Engineering"},
    {"id": 2, "department": "Sales"},
    {"id": 4, "department": "Marketing"},  # No matching user!
]

print("Users table:")
for u in users:
    print(f"  {u}")

print("\nDepartments table:")
for d in departments:
    print(f"  {d}")

result = left_join(users, departments, 'id')

print("\n📊 LEFT JOIN Result:")
for row in result:
    dept = row.get('department', 'NULL')  # NULL if no match
    print(f"  {row['name']}: {dept}")

print("\n💡 Charlie has no department (LEFT JOIN keeps all left rows)")
print("💡 Marketing is dropped (no matching user in left table)")

---
## Summary: Key Data Structure Takeaways

| Concept | When to Use | Python Tool |
|---------|-------------|-------------|
| Flatten/Unflatten | API JSON, Spark prep | Recursion |
| Counting | Aggregations | `defaultdict(int)` |
| Grouping | Category analysis | `defaultdict(list)` |
| Large Files | Memory efficiency | Generators (`yield`) |
| Rolling Metrics | Time series | `deque(maxlen=n)` |
| Fast Lookups | JOINs, deduplication | Dict/Hash map |